# Install pyspark

In [ ]:
!pip install pyspark

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.4.0-py2.py3-none-any.whl size=311317145 sha256=2e33692f0b8d867e24c3aa9b4ffe3f13b00e370ee382dbef4d585885e0893ac6
  Stored in directory: /root/.cache/pip/wheels/7b/1b/4b/3363a1d04368e7ff0d408e57ff57966fcdf00583774e761327
Successfully built pyspark


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Montage drive (optionnel)
from google.colab import drive
drive.mount('/My_drive/')

Mounted at /My_drive/


# TP numero 1

In [ ]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
sc

<SparkContext master=local[*] appName=pyspark-shell>

In [ ]:
#Charger un csv (modifier le path)
path="/My_drive/My Drive/Cours prof Big data/"
df = spark.read.csv(path+"vgsales.csv",header=True)
#Compter le nombre de lignes
print(df.count())
#Afficher les 20 premières lignes
df.show()

16598
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|Rank|                Name|Platform|Year|       Genre|           Publisher|NA_Sales|EU_Sales|JP_Sales|Other_Sales|Global_Sales|
+----+--------------------+--------+----+------------+--------------------+--------+--------+--------+-----------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   41.49|   29.02|    3.77|       8.46|       82.74|
|   2|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|   29.08|    3.58|    6.81|       0.77|       40.24|
|   3|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   15.85|   12.88|    3.79|       3.31|       35.82|
|   4|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   15.75|   11.01|    3.28|       2.96|          33|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|   11.27|    8.89|   10

<h1> Compter le nombre de jv Nintendo </h1>

In [ ]:
#Via un DF
df_filter = df.where(df.Publisher=="Nintendo")
df_filter.count()

703

In [ ]:
#Via un RDD
df.rdd.filter(lambda x : x.Publisher=="Nintendo").count()

703

In [ ]:
#Via du SQL
df.createOrReplaceTempView("jv_vente")
requete_sql = spark.sql("SELECT count(*) FROM jv_vente where jv_vente.Publisher=='Nintendo'")
requete_sql.show()

+--------+
|count(1)|
+--------+
|     703|
+--------+



# Afficher les types de vos colonnes dans votre dataframe

# Compter le nombre de jeux à plus de 10 millions de ventes globales
Nb : si vous obtenez 52, vous avez oublié une étape

# Afficher les 40 première lignes avec uniquement les colonnes Name,Platform et Year



<h1>Rajouter une colonne concernant le pourcentage de vente EU </h1>

In [ ]:
#Via la methode withColumn


In [ ]:
#Via du SQL


+--------------------+--------+----+------------+--------------------+--------+------------+------------------+
|                Name|Platform|Year|       Genre|           Publisher|EU_Sales|Global_Sales|    pourcentage_EU|
+--------------------+--------+----+------------+--------------------+--------+------------+------------------+
|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   29.02|       82.74| 35.07372638025055|
|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|    3.58|       40.24| 8.896619717642263|
|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   12.88|       35.82|  35.9575662316435|
|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   11.01|          33| 33.36363705721768|
|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|    8.89|       31.37|28.339177894456196|
|              Tetris|      GB|1989|      Puzzle|            Nintendo|    2.26|       30.26| 7.468605331

# Pour exemple : ajouter une colonne concernant le pourcentage de vente EU Methode complexe (avec UDF ou RDD)

In [ ]:
def calcul_pourcentage(sales,global_sales) :
    return float(sales)/float(global_sales)*100

calcul_pourcentage("1","49")

2.0408163265306123

In [ ]:
#Via un Dataframe
def calcul_pourcentage(sales,global_sales, multiply_by_cent = True) :
  if multiply_by_cent :
    return float(sales)/float(global_sales)*100
  else :
    return float(sales)/float(global_sales)

df_prop= df.withColumn('pourcentage_EU',
                       F.udf(
                           lambda a,b : calcul_pourcentage(a,b, False),
                           #calcul_pourcentage,
                           T.FloatType()
                           )("EU_Sales","Global_Sales"))

df_prop.select(["Name","Platform","Year","Genre","Publisher","EU_Sales","Global_Sales","pourcentage_EU"]).show()

+--------------------+--------+----+------------+--------------------+--------+------------+--------------+
|                Name|Platform|Year|       Genre|           Publisher|EU_Sales|Global_Sales|pourcentage_EU|
+--------------------+--------+----+------------+--------------------+--------+------------+--------------+
|          Wii Sports|     Wii|2006|      Sports|            Nintendo|   29.02|       82.74|    0.35073724|
|   Super Mario Bros.|     NES|1985|    Platform|            Nintendo|    3.58|       40.24|   0.088966206|
|      Mario Kart Wii|     Wii|2008|      Racing|            Nintendo|   12.88|       35.82|    0.35957566|
|   Wii Sports Resort|     Wii|2009|      Sports|            Nintendo|   11.01|          33|    0.33363637|
|Pokemon Red/Pokem...|      GB|1996|Role-Playing|            Nintendo|    8.89|       31.37|    0.28339177|
|              Tetris|      GB|1989|      Puzzle|            Nintendo|    2.26|       30.26|    0.07468606|
|New Super Mario B...|      

<h1> Afficher les différentes plateformes</h1>

<h1> Afficher les ventes de jeux vidéo par année (ordonner les années) </h1>

<h1>Rajouter une colonne pour indiquer si le jeux est un jeux pokémon (commencez simple et complexifier)</h1>

In [ ]:
#Attention le texte est un format libre


<h1> Exemple : Quels sont les 20 mots les plus présent </h1>

In [ ]:
df_1 = df.withColumn("words",F.udf(lambda x :x.lower().split(),\
                                   T.ArrayType(T.StringType()))("Name"))
print(df_1.first())
df_2 =df_1.withColumn("word",F.explode("words"))
print(df_2.take(2))
df_3 = df_2.groupBy("word").count().orderBy("count",ascending=False)
df_3.show()

Row(Rank='1', Name='Wii Sports', Platform='Wii', Year='2006', Genre='Sports', Publisher='Nintendo', NA_Sales='41.49', EU_Sales='29.02', JP_Sales='3.77', Other_Sales='8.46', Global_Sales='82.74', words=['wii', 'sports'])
[Row(Rank='1', Name='Wii Sports', Platform='Wii', Year='2006', Genre='Sports', Publisher='Nintendo', NA_Sales='41.49', EU_Sales='29.02', JP_Sales='3.77', Other_Sales='8.46', Global_Sales='82.74', words=['wii', 'sports'], word='wii'), Row(Rank='1', Name='Wii Sports', Platform='Wii', Year='2006', Genre='Sports', Publisher='Nintendo', NA_Sales='41.49', EU_Sales='29.02', JP_Sales='3.77', Other_Sales='8.46', Global_Sales='82.74', words=['wii', 'sports'], word='sports')]
+------+-----+
|  word|count|
+------+-----+
|   the| 2733|
|    of| 1718|
|     2|  844|
|    no|  743|
|     3|  399|
| world|  386|
|     &|  353|
|    2:|  323|
|   pro|  320|
|  game|  300|
|    to|  289|
| super|  289|
|   and|  260|
|     -|  248|
|  star|  235|
|soccer|  227|
|dragon|  214|
|    ii|  

<h1> Quel mot fait le plus vendre ? </h1>
NB : vous pouvez vous insiprer du code précédent



<h1> Partie Machine-learning </h1>

<h1> A partir des différentes colonnes (sans les ventes globales) essayez de prédire les ventes européennes </h1>

<h1> Evaluez votre algorithme et essayez en d'autres </h1>
Reflechissez à la methode que vous voulez utiliser

<h1> Rajouter des colonnes pour améliorer vos algorithmes </h1>